# 🔬 02 - Feature Engineering, Linguistic Signals & Hype Scores

### Mathematical & Linguistic Architecture of the BS & Hype Metric

Financial and media sensationalism operates across five quantifiable linguistic dimensions:

1. **Superlative / Buzzword Density ($S_{sup}$)**: Density of hyperbolic intensifiers (`game-changer`, `skyrocket`, `to the moon`, `bloodbath`, `100x`).
2. **Sentiment Subjectivity ($S_{subj}$)**: Polarity subjectivity metric ($0.0 = \text{objective fact}$, $1.0 = \text{pure opinion}$). 
3. **Clickbait Syntactic Signals ($S_{click}$)**: Headline all-caps ratio, exclamation intensity, urgency triggers (`urgent warning`, `must watch`).
4. **Vague Authority Citations ($S_{vague}$)**: Anonymous assertions without named sources (`experts warn`, `insiders suggest`, `market whispers`).
5. **Quantitative Grounding Disparity ($S_{unsub}$)**: Disproportion between qualitative claims and verified empirical stats/currencies/numbers.

$$\text{Hype Score} = \text{clamp}\Big(\sum w_i S_i \cdot (0.8 + 0.2 \cdot \Omega_{\text{outlet}}), 0.0, 1.0\Big)$$

---

In [ ]:
import pandas as pd
from src.config import INTERIM_DATA_DIR, PROCESSED_DATA_DIR
from src.features import HypeFeatureExtractor
from src.viz import (
    plot_hype_distribution,
    plot_outlet_comparison,
    plot_subjectivity_vs_hype,
    plot_hype_dimension_radar
)

# Load ingested data
df_raw = pd.read_csv(INTERIM_DATA_DIR / 'ingested_articles.csv')

# Run Feature Extraction Engine
extractor = HypeFeatureExtractor()
df_features = extractor.process_dataframe(df_raw)

print(f'Processed {len(df_features)} media items.')
df_features[['title', 'outlet', 'subjectivity_score', 'superlative_density', 'clickbait_syntax', 'hype_score', 'hype_tier']].head(5)

## 2. Hype Score Distribution Across the Media Spectrum

Let's visualize the composite score distribution and evaluate our decision threshold ($	au = 0.50$).

In [ ]:
fig_dist = plot_hype_distribution(df_features, threshold=0.50)
fig_dist.show()

## 3. Outlet Comparative Ranking

Which media outlets and channels exhibit the highest concentration of sensationalism?

In [ ]:
fig_outlets = plot_outlet_comparison(df_features)
fig_outlets.show()

## 4. 2D Diagnostic: Subjectivity vs. Hype Density

In this view, bubble size represents syntactic clickbait intensity. Notice how wire services (Reuters, WSJ, Bloomberg) cluster in the lower left corner (low subjectivity, low hype), whereas sensational broadcast and crypto influencers dominate the upper right quadrant.

In [ ]:
fig_scatter = plot_subjectivity_vs_hype(df_features)
fig_scatter.show()

## 5. Case Study: 5-Factor Linguistic Diagnostics

Compare the diagnostic radar fingerprint of the most sensational item vs an objective wire release.

In [ ]:
top_hyped = df_features.sort_values(by='hype_score', ascending=False).iloc[0].to_dict()
most_objective = df_features.sort_values(by='hype_score', ascending=True).iloc[0].to_dict()

fig_radar_hyped = plot_hype_dimension_radar(top_hyped)
fig_radar_hyped.show()

fig_radar_obj = plot_hype_dimension_radar(most_objective)
fig_radar_obj.show()

## 6. Persist Processed Features to Parquet & CSV

Export processed features to `data/processed/sample_features.parquet` for downstream graph and dashboard use.

In [ ]:
df_features.to_parquet(PROCESSED_DATA_DIR / 'sample_features.parquet', index=False)
df_features.to_csv(PROCESSED_DATA_DIR / 'sample_features.csv', index=False)
print('✓ Processed features saved to Parquet and CSV. Proceed to Notebook 03!')